In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier

In [26]:
df = pd.read_parquet(
    "../data/processed/cybersecurity_ml_ready.parquet"
)

df.head()

,protocol,packet_length,traffic_type,anomaly_scores,severity_level,network_segment,mexico_state,attack_type
0,0,503,2,28.67,1,0,3,2
1,0,1174,2,51.50,1,1,7,2
2,2,306,2,87.42,1,2,9,0
3,2,385,2,15.79,2,1,9,2
4,1,1462,0,0.52,1,2,11,0


In [27]:
df["high_anomaly"] = (
    df["anomaly_scores"] > 70
).astype(int)

df["critical_packet"] = (
    df["packet_length"] > 1000
).astype(int)

df["severity_encoded"] = (
    df["severity_level"] >= 1
).astype(int)

df.head()

,protocol,packet_length,traffic_type,anomaly_scores,severity_level,network_segment,mexico_state,attack_type,high_anomaly,critical_packet,severity_encoded
0,0,503,2,28.67,1,0,3,2,0,0,1
1,0,1174,2,51.50,1,1,7,2,0,1,1
2,2,306,2,87.42,1,2,9,0,1,0,1
3,2,385,2,15.79,2,1,9,2,0,0,1
4,1,1462,0,0.52,1,2,11,0,0,1,1


In [28]:
print(df.columns)

print(df.shape)

Index(['protocol', 'packet_length', 'traffic_type', 'anomaly_scores',
       'severity_level', 'network_segment', 'mexico_state', 'attack_type',
       'high_anomaly', 'critical_packet', 'severity_encoded'],
      dtype='str')
(40000, 11)


In [29]:
X = df.drop("attack_type", axis=1)
y = df["attack_type"]

print(X.shape)
print(y.shape)

(40000, 10)
(40000,)


In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(32000, 10)
(8000, 10)


In [31]:
cat_model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",
    verbose=50,
    random_state=42
)

cat_model.fit(X_train, y_train)

0:	learn: 1.0983962	total: 16.7ms	remaining: 3.32s
50:	learn: 1.0868616	total: 293ms	remaining: 855ms
100:	learn: 1.0785160	total: 596ms	remaining: 584ms
150:	learn: 1.0707105	total: 859ms	remaining: 279ms
199:	learn: 1.0635793	total: 1.13s	remaining: 0us


CatBoostClassifier(depth=6, iterations=200, learning_rate=0.1, loss_function='MultiClass', random_state=42, verbose=50)

In [32]:
y_pred_cat = cat_model.predict(X_test)

print(y_pred_cat[:10])

[[2]
 [0]
 [1]
 [0]
 [0]
 [0]
 [0]
 [2]
 [2]
 [1]]


In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

cat_accuracy = accuracy_score(
    y_test,
    y_pred_cat
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred_cat,
        average="weighted"
    )
)

In [34]:
cat_accuracy = accuracy_score(
    y_test,
    y_pred_cat
)

print("CatBoost Accuracy:", cat_accuracy)

CatBoost Accuracy: 0.3245


In [35]:
print(
    classification_report(
        y_test,
        y_pred_cat
    )
)

              precision    recall  f1-score   support

           0       0.32      0.34      0.33      2636
           1       0.33      0.29      0.31      2721
           2       0.32      0.35      0.33      2643

    accuracy                           0.32      8000
   macro avg       0.32      0.32      0.32      8000
weighted avg       0.32      0.32      0.32      8000



In [36]:
cat_cm = confusion_matrix(
    y_test,
    y_pred_cat
)

print(cat_cm)

[[904 807 925]
 [949 778 994]
 [936 793 914]]


In [37]:
import json
import os

os.makedirs(
    "../data/metrics",
    exist_ok=True
)

cat_metrics = {
    "model": "CatBoost",
    "accuracy": float(cat_accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1)
}

with open(
    "../data/metrics/catboost_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        cat_metrics,
        f,
        indent=4
    )

print("CatBoost metrics exported")

CatBoost metrics exported
